<a href="https://colab.research.google.com/github/rathore-rl1lmg1/Cooding-Playground/blob/main/Untitled8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gymnasium as gym
import numpy as np
import random
import torch
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torch import nn
import torch.optim as optim
from collections import deque
import torch.nn as nn

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

In [ ]:
obs , info = env.reset()

In [ ]:
action = (0,1) # 0=left push 1=right push
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class DQN:
    def __init__(self,env,gamma,epsilon,num_epi):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.num_epi = num_epi
        self.state_dimn=4  # pos , vel , angle , w
        self.action_dimn=2 # left push=0, right push=1
        #max size of replay buffer
        replay_buffer_size = 5000
        #batch size randomly taken from buffer
        self.batch_rep_buff = 100
        #num training episode before target update
        self.target_period = 100
        #counter to count when target reached
        self.count_target_period = 0
        #sum of reward obtained
        self.net_reward=[]
        self.replay_buffer = deque(maxlen=replay_buffer_size)
        #creat nn
        self.main_network = NeuralNet().to(device)   #online
        self.target_network = NeuralNet().to(device) #target
        #take initial weights to target
        self.target_network.load_state_dict(self.main_network.state_dict())

        self.optimizer = optim.Adam(self.main_network.parameters(), lr=0.0005)
        self.criterion = nn.MSELoss()
         # freeze target network
        for param in self.target_network.parameters():
            param.requires_grad = False
        #
        self.action_append =[]

    def my_loss_fn(self, y_true, y_pred, actions):
        # reshape actions
        actions = actions.unsqueeze(1)

        # select Q-values for taken actions
        pred_q = y_pred.gather(1, actions)   # (batch, 1)
        true_q = y_true.gather(1, actions)   # (batch, 1)

        # MSE loss
        loss = self.criterion(pred_q, true_q)

        return loss

    def train_epi(self):
        for epi in range(self.num_epi):
            reward_epi = []

            #print("simulating epi {}".format(epi))
            #reset env
            (current_state,_)=self.env.reset()
            #initialize state
            terminal_state = False
            while not terminal_state:
                #select action based on curr_state
                action = self.select_action(current_state,epi)

                #take step and return new state, reward & is the state terminal
                (next_state, reward, terminal_state,_,_)=self.env.step(action)
                reward_epi.append(reward)

                #add current state action and rest to the replay buffer
                self.replay_buffer.append((current_state,action,reward,next_state,terminal_state))

                #train network
                if len(self.replay_buffer) > 1000:
                    self.train_network()

                #set next state as current state
                current_state = next_state
            print("sum of rewards {}".format(np.sum(reward_epi)))
            self.net_reward.append(np.sum(reward_epi))
    def select_action(self,state,epi):
        if epi < 1:
            return np.random.choice(self.action_dimn) #first epi every action randomly taken

        rand_num = np.random.random() #for epsilon greedy approach

        #after index epi slowly dec epsilon
        if epi > 200:
            self.epsilon = 0.995*self.epsilon

        if rand_num < self.epsilon:
            return np.random.choice(self.action_dimn)

        else:
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                Qvalues = self.main_network(state_tensor)

            action = self.main_network(state_tensor)
            return torch.argmax(Qvalues).item()

    def train_network(self):
        if len(self.replay_buffer) > self.batch_rep_buff:

            # sample batch
            batch = random.sample(self.replay_buffer, self.batch_rep_buff)

            # unpack batch
            states, actions, rewards, next_states, dones = zip(*batch)

            # convert to tensors
            states = torch.tensor(states, dtype=torch.float32).to(device)
            next_states = torch.tensor(next_states, dtype=torch.float32).to(device)
            actions = torch.tensor(actions, dtype=torch.long).to(device)
            rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
            dones = torch.tensor(dones, dtype=torch.float32).to(device)

            # Q(s,a) from main network
            curr_q = self.main_network(states)

            # Q(s',a') from target network
            with torch.no_grad():
                next_q = self.target_network(next_states)

            # max Q for next state
            max_next_q = torch.max(next_q, dim=1)[0]

            # compute target
            target_q = rewards + self.gamma * max_next_q * (1 - dones)

            # select Q-values for taken actions
            curr_q_selected = curr_q.gather(1, actions.unsqueeze(1)).squeeze()

            # loss
            loss = F.mse_loss(curr_q_selected, target_q)

            # backprop
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            # update target network
            self.count_target_period += 1

            if self.count_target_period >= self.target_period:
                self.target_network.load_state_dict(self.main_network.state_dict())
                #if epi % 100 ==0:
                #print("Target network updated!")
                #print(f"Counter value {self.count_target_period}")
                self.count_target_period = 0



In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 56),
            nn.ReLU(),
            nn.Linear(56, 2)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
gamma = 1
epsilon = 0.1

num_epi = 1000
learn_DQN = DQN(env, gamma, epsilon, num_epi) #creat
learn_DQN.train_epi()
learn_DQN.net_reward

In [ ]:
from torchsummary import summary

summary(learn_DQN.main_network, (4,))

In [ ]:
torch.save(learn_DQN.main_network.state_dict(), "trained_model.pth")

In [ ]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# create model
model = NeuralNet().to(device)

# load weights
model.load_state_dict(torch.load("trained_model.pth", map_location=device))
model.eval()   # VERY IMPORTANT

sum_reward = 0

# create env
env = gym.make("CartPole-v1", render_mode='rgb_array')
env = gym.wrappers.RecordVideo(
    env,
    video_folder="stored_video",
    episode_trigger=lambda x: True
)

(current_state, _) = env.reset()


terminal_state = False

while not terminal_state:

    # to tensor
    state_tensor = torch.tensor(current_state, dtype=torch.float32).unsqueeze(0).to(device)

    # get Q-values
    with torch.no_grad():
        Qvalues = model(state_tensor)

    # choose best action
    action = torch.argmax(Qvalues).item()

    # take step
    (current_state, reward, terminal_state, _, _) = env.step(action)

    sum_reward += reward

env.close()

print("Total Reward:", sum_reward)

In [ ]:
from IPython.display import Video
Video("stored_video/rl-video-episode-0.mp4", embed=True)

In [ ]:
import os
os.listdir("stored_video")
